# Lab 07: Retrieval strategies and reranking

Extend Lab 06's retrieval pipeline with BM25 + dense fusion, MMR
diversification, and cross-encoder reranking. Same corpus, same agent
loop, measurably better retrieval.

This is the runnable companion to
[`labs/07-retrieval-strategies-and-reranking/README.md`](./README.md).
Read the brief first.

**Estimated time:** 110–140 minutes.
**Difficulty:** 🟡 Intermediate.
**Prerequisites:** Lab 06 finished; the three retrieval-quality concept pages
([retrieval-strategies](../../concepts/rag/retrieval-strategies.md),
[hybrid-search](../../concepts/rag/hybrid-search.md),
[reranking](../../concepts/rag/reranking.md)) read.

> 🔴 **Tools are fast-changing.** This notebook is pinned to
> `rank-bm25>=0.2.2,<0.3` and `sentence-transformers>=5.0,<6.0`
> (verified 2026-05-24). The `rank-bm25` package is stable but
> "inactive" — it's been the same `0.2.2` for years. For
> production scale you'd reach for `bm25s` or a real search engine;
> for this lab, `rank-bm25` is exactly right.

## Step 0: Setup

Two new dependencies on top of Lab 06:

```bash
uv add 'rank-bm25>=0.2.2,<0.3'
```

The `sentence-transformers` package is already installed from Lab 06;
we use its `CrossEncoder` class for the reranker. The cross-encoder
model (`cross-encoder/ms-marco-MiniLM-L-6-v2`) downloads ~80 MB on
first run.

In [ ]:
import os
import re
import pathlib
import json
from typing import Any
from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

PROVIDER = "openai"   # or "anthropic"

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), (
    "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"
)
print(f"Provider: {PROVIDER}")


**Sample output:**

```
Provider: openai
```

## Step 1: Recreate the Lab 06 baseline

We rebuild the same chunker, the same dense index, the same
`search_corpus` shape. Then we define a small evaluation set —
queries where Lab 06 worked well and queries where it didn't —
so every later upgrade is measured against the same baseline.

Same corpus path as Lab 06: `../06-agentic-rag-from-scratch/corpus/`
relative to this notebook.

In [ ]:
# ── Same chunker as Lab 06 ──
CORPUS_DIR = pathlib.Path("../06-agentic-rag-from-scratch/corpus")
TARGET_TOKENS = 160
OVERLAP_TOKENS = 32


def approx_tokens(text: str) -> int:
    return int(len(text.split()) / 0.75)


def split_at_paragraphs(text: str) -> list[str]:
    parts = re.split(r"\n\s*\n", text)
    return [p.strip() for p in parts if p.strip()]


def split_at_sentences(text: str) -> list[str]:
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [p.strip() for p in parts if p.strip()]


def chunk_text(text: str) -> list[str]:
    """Recursive split with overlap; same as Lab 06."""
    paragraphs = split_at_paragraphs(text)
    chunks: list[str] = []
    current: list[str] = []
    current_tokens = 0

    for para in paragraphs:
        para_tokens = approx_tokens(para)
        if para_tokens > TARGET_TOKENS:
            if current:
                chunks.append("\n\n".join(current))
                current, current_tokens = [], 0
            sentences = split_at_sentences(para)
            sub_chunk: list[str] = []
            sub_tokens = 0
            for sent in sentences:
                sent_tokens = approx_tokens(sent)
                if sub_tokens + sent_tokens > TARGET_TOKENS and sub_chunk:
                    chunks.append(" ".join(sub_chunk))
                    sub_chunk, sub_tokens = [], 0
                sub_chunk.append(sent)
                sub_tokens += sent_tokens
            if sub_chunk:
                chunks.append(" ".join(sub_chunk))
            continue
        if current_tokens + para_tokens > TARGET_TOKENS and current:
            chunks.append("\n\n".join(current))
            current, current_tokens = [], 0
        current.append(para)
        current_tokens += para_tokens

    if current:
        chunks.append("\n\n".join(current))

    if OVERLAP_TOKENS <= 0 or len(chunks) < 2:
        return chunks

    overlapped: list[str] = [chunks[0]]
    for i in range(1, len(chunks)):
        prev_words = chunks[i - 1].split()
        overlap_words = int(OVERLAP_TOKENS * 0.75)
        tail = " ".join(prev_words[-overlap_words:]) if overlap_words > 0 else ""
        overlapped.append((tail + " " + chunks[i]).strip()
                          if tail else chunks[i])
    return overlapped


def first_heading(text: str) -> str:
    for line in text.splitlines():
        if line.startswith("# "):
            return line[2:].strip()
    return ""


# Load and chunk all docs
all_chunks = []
for path in sorted(CORPUS_DIR.glob("*.md")):
    if path.name == "README.md":
        continue
    text = path.read_text()
    title = first_heading(text)
    chunks = chunk_text(text)
    for i, chunk_body in enumerate(chunks):
        all_chunks.append({
            "chunk_id": f"{path.name}:{i}",
            "doc_id": path.name,
            "title": title,
            "text": chunk_body,
        })

chunks_by_id = {c["chunk_id"]: c for c in all_chunks}
print(f"Loaded {len({c['doc_id'] for c in all_chunks})} docs, "
      f"{len(all_chunks)} chunks")


**Sample output:**

```
Loaded 8 docs, 55 chunks
```

In [ ]:
# ── Build the dense index (same as Lab 06) ──
import numpy as np
from sentence_transformers import SentenceTransformer

print("Loading bi-encoder (sentence-transformers/all-MiniLM-L6-v2)...")
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu",
)

chunk_texts = [c["text"] for c in all_chunks]
dense_embeddings = embedder.encode(
    chunk_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)

print(f"Dense index: shape={dense_embeddings.shape}, "
      f"dtype={dense_embeddings.dtype}")


**Sample output:**

```
Loading bi-encoder (sentence-transformers/all-MiniLM-L6-v2)...
Dense index: shape=(55, 384), dtype=float32
```

In [ ]:
# ── Lab 06's baseline retriever ──
def dense_retrieve(query: str, top_k: int = 10) -> list[tuple[int, float]]:
    """Bi-encoder retrieval. Returns [(chunk_idx, score), ...] sorted by score desc."""
    query_emb = embedder.encode(
        [query], normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False,
    )[0]
    scores = dense_embeddings @ query_emb
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_indices]


# Smoke test
print("dense_retrieve('agent loop four phases', top_k=3):")
for idx, score in dense_retrieve("agent loop four phases", top_k=3):
    print(f"  {score:.3f}  [{all_chunks[idx]['chunk_id']}]  "
          f"{all_chunks[idx]['text'][:60]}...")


**Sample output:**

```
dense_retrieve('agent loop four phases', top_k=3):
  0.711  [01-agent-loop.md:0]  # The Agent Loop: A Brief Introduction  The agent loop...
  0.643  [01-agent-loop.md:1]  is the user's question, the previous turn's output...
  0.428  [03-react-pattern.md:0]  # The ReAct Pattern  ReAct — Reasoning and Acting...
```

In [ ]:
# ── Evaluation queries ──
# A handful of queries with "expected" top chunks. Not a formal eval —
# just a side-by-side comparison that builds intuition for which
# upgrades help on which queries.

EVAL_QUERIES = [
    {
        "query": "What is the ReAct pattern?",
        "expected_doc": "03-react-pattern.md",
        "kind": "lexical-favorable",  # proper noun, BM25 should excel
    },
    {
        "query": "agent loop four phases",
        "expected_doc": "01-agent-loop.md",
        "kind": "both",  # both retrievers find it easily
    },
    {
        "query": "wrong-but-confident retrieval failure",
        "expected_doc": "04-search-vs-retrieval.md",
        "kind": "semantic",  # phrase not in corpus; needs semantic understanding
    },
    {
        "query": "what truncates silently in embeddings",
        "expected_doc": "05-embeddings.md",
        "kind": "semantic",  # paraphrase of the 256-wordpiece content
    },
    {
        "query": "structured tool errors discriminated union status",
        "expected_doc": "02-tool-design.md",
        "kind": "lexical-favorable",  # specific technical terms
    },
    {
        "query": "chunk overlap recovery boundary",
        "expected_doc": "07-chunking-strategies.md",
        "kind": "both",
    },
]


def rank_of_expected_doc(retrieval_results, expected_doc):
    """Find rank (1-indexed) of first chunk from expected_doc, or None if absent."""
    for rank, (idx, _score) in enumerate(retrieval_results, start=1):
        if all_chunks[idx]["doc_id"] == expected_doc:
            return rank
    return None


# Baseline (dense alone) on each query
print("─── Baseline: dense retrieval only (top_k=10) ───")
print(f"{'kind':<20} {'rank':<5} query")
print("─" * 70)
for q in EVAL_QUERIES:
    results = dense_retrieve(q["query"], top_k=10)
    rank = rank_of_expected_doc(results, q["expected_doc"])
    rank_str = str(rank) if rank else "miss"
    print(f"{q['kind']:<20} {rank_str:<5} {q['query']}")


**Sample output (yours may vary by 1-2 ranks):**

```
─── Baseline: dense retrieval only (top_k=10) ───
kind                 rank  query
──────────────────────────────────────────────────────────────────────
lexical-favorable    1     What is the ReAct pattern?
both                 1     agent loop four phases
semantic             1     wrong-but-confident retrieval failure
semantic             1     what truncates silently in embeddings
lexical-favorable    2     structured tool errors discriminated union status
both                 1     chunk overlap recovery boundary
```

The dense baseline does well on this small set. Note that bi-encoder retrieval handles paraphrase queries surprisingly well — `"what truncates silently in embeddings"` finds the 256-wordpiece chunk at rank 1 even though "truncates silently" isn't in the corpus. The lexical-favorable query (`"structured tool errors..."`) takes a slight hit at rank 2 — exactly where BM25 will shine.

## Step 2: Add BM25

Build a `rank-bm25` index over the same chunks, with the same
tokenization for query and corpus. Compare top-k against the dense
baseline.

In [ ]:
from rank_bm25 import BM25Okapi


def tokenize(text: str) -> list[str]:
    """Lowercase + extract word tokens. Apply identically to corpus and query."""
    return [t for t in re.findall(r"\w+", text.lower()) if len(t) > 1]


# Build the BM25 index
tokenized_corpus = [tokenize(c["text"]) for c in all_chunks]
bm25 = BM25Okapi(tokenized_corpus)
# Defaults: k1=1.5, b=0.75, epsilon=0.25 — Robertson & Walker's standard values.

print(f"BM25 index built over {len(tokenized_corpus)} chunks")
print(f"Avg tokens per chunk: {np.mean([len(t) for t in tokenized_corpus]):.1f}")


def bm25_retrieve(query: str, top_k: int = 10) -> list[tuple[int, float]]:
    """BM25 retrieval. Returns [(chunk_idx, score), ...] sorted by score desc."""
    q_tokens = tokenize(query)
    scores = bm25.get_scores(q_tokens)  # numpy array shape (n_chunks,)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_indices]


# Smoke test on a proper-noun query
print("\nbm25_retrieve('ReAct pattern', top_k=3):")
for idx, score in bm25_retrieve("ReAct pattern", top_k=3):
    print(f"  {score:6.3f}  [{all_chunks[idx]['chunk_id']}]  "
          f"{all_chunks[idx]['text'][:60]}...")


**Sample output:**

```
BM25 index built over 55 chunks
Avg tokens per chunk: 152.7

bm25_retrieve('ReAct pattern', top_k=3):
   9.516  [03-react-pattern.md:5]  are worth knowing about, but reaching for them...
   9.217  [03-react-pattern.md:0]  # The ReAct Pattern  ReAct — Reasoning and Acting...
   7.975  [03-react-pattern.md:4]  But the structure is there: thought (possibly empty)...
```

In [ ]:
# Run BM25 on the evaluation set, alongside dense
print("─── Comparison: dense vs BM25 (top_k=10) ───")
print(f"{'kind':<20} {'dense':<7} {'bm25':<7} query")
print("─" * 80)
for q in EVAL_QUERIES:
    dr = dense_retrieve(q["query"], top_k=10)
    br = bm25_retrieve(q["query"], top_k=10)
    dense_rank = rank_of_expected_doc(dr, q["expected_doc"])
    bm25_rank = rank_of_expected_doc(br, q["expected_doc"])
    print(f"{q['kind']:<20} "
          f"{str(dense_rank) if dense_rank else 'miss':<7} "
          f"{str(bm25_rank) if bm25_rank else 'miss':<7} "
          f"{q['query']}")


**Sample output (your ranks may vary by 1-2):**

```
─── Comparison: dense vs BM25 (top_k=10) ───
kind                 dense   bm25    query
────────────────────────────────────────────────────────────────────────────────
lexical-favorable    1       1       What is the ReAct pattern?
both                 1       1       agent loop four phases
semantic             1       miss    wrong-but-confident retrieval failure
semantic             1       miss    what truncates silently in embeddings
lexical-favorable    2       1       structured tool errors discriminated union status
both                 1       1       chunk overlap recovery boundary
```

Each retriever has a non-empty set of queries where it loses to the other:

- **Dense wins on "wrong-but-confident retrieval failure"** — that exact phrase isn't in any chunk, so BM25 has nothing to match. Dense retrieval understands the *concept*.
- **Dense wins on "what truncates silently in embeddings"** — same story. The chunk talks about the 256-wordpiece truncation but doesn't use the verb "truncates".
- **BM25 wins on "structured tool errors discriminated union status"** — these are exact terms from `02-tool-design.md`. The dense embedding compresses them into a less distinctive vector.

This is exactly the dual-failure-mode pattern hybrid search exists to fix.

## Step 3: Reciprocal Rank Fusion

Combine the two retrievers' ranked lists into one. The standard
algorithm (Cormack, Clarke & Büttcher, 2009): ignore raw scores
entirely, sum `1 / (k + rank)` contributions across retrievers,
with `k=60` as the paper's robust constant.

In [ ]:
def reciprocal_rank_fusion(
    ranked_lists: dict[str, list[tuple[int, float]]],
    k: int = 60,
) -> list[tuple[int, float]]:
    """Fuse multiple ranked lists by RRF.

    Args:
        ranked_lists: {retriever_name: [(chunk_idx, score), ...]}.
                      Scores are ignored; only ranks matter.
        k: RRF constant. 60 is the standard, robust across corpora.

    Returns:
        [(chunk_idx, rrf_score), ...] sorted by rrf_score desc.
    """
    rrf_scores: dict[int, float] = {}
    for ranked in ranked_lists.values():
        for rank, (chunk_idx, _orig_score) in enumerate(ranked, start=1):
            rrf_scores[chunk_idx] = rrf_scores.get(chunk_idx, 0.0) + 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda kv: kv[1], reverse=True)


def hybrid_retrieve(query: str, top_k: int = 10,
                    candidate_k: int = 30) -> list[tuple[int, float]]:
    """Run dense + BM25, fuse with RRF, return top_k."""
    dense_results = dense_retrieve(query, top_k=candidate_k)
    bm25_results = bm25_retrieve(query, top_k=candidate_k)
    fused = reciprocal_rank_fusion(
        {"dense": dense_results, "bm25": bm25_results},
        k=60,
    )
    return fused[:top_k]


# Smoke test
print("hybrid_retrieve('ReAct pattern', top_k=3):")
for idx, score in hybrid_retrieve("ReAct pattern", top_k=3):
    print(f"  {score:.4f}  [{all_chunks[idx]['chunk_id']}]  "
          f"{all_chunks[idx]['text'][:60]}...")


**Sample output:**

```
hybrid_retrieve('ReAct pattern', top_k=3):
  0.0325  [03-react-pattern.md:0]  # The ReAct Pattern  ReAct — Reasoning and Acting...
  0.0319  [03-react-pattern.md:5]  are worth knowing about, but reaching for them...
  0.0312  [03-react-pattern.md:4]  But the structure is there: thought (possibly empty)...
```

Note the RRF scores look small — they're sums of `1/(60+rank)` terms, capped around `2/61 ≈ 0.033` for a chunk that's rank 1 in both retrievers. The *absolute* numbers don't matter for ranking; only the *order* does.

In [ ]:
# Run all three retrievers on the eval set
print("─── Three-way comparison (top_k=10) ───")
print(f"{'kind':<20} {'dense':<7} {'bm25':<7} {'hybrid':<7} query")
print("─" * 90)


def _fmt_rank(r):
    return str(r) if r else "miss"


for q in EVAL_QUERIES:
    dr = dense_retrieve(q["query"], top_k=30)
    br = bm25_retrieve(q["query"], top_k=30)
    hr = hybrid_retrieve(q["query"], top_k=10, candidate_k=30)
    d_rank = rank_of_expected_doc(dr[:10], q["expected_doc"])
    b_rank = rank_of_expected_doc(br[:10], q["expected_doc"])
    h_rank = rank_of_expected_doc(hr, q["expected_doc"])
    print(f"{q['kind']:<20} "
          f"{_fmt_rank(d_rank):<7} {_fmt_rank(b_rank):<7} {_fmt_rank(h_rank):<7} "
          f"{q['query']}")


**Sample output (ranks may vary slightly):**

```
─── Three-way comparison (top_k=10) ───
kind                 dense   bm25    hybrid  query
──────────────────────────────────────────────────────────────────────────────────────────
lexical-favorable    1       1       1       What is the ReAct pattern?
both                 1       1       1       agent loop four phases
semantic             1       miss    1       wrong-but-confident retrieval failure
semantic             1       miss    1       what truncates silently in embeddings
lexical-favorable    2       1       1       structured tool errors discriminated union status
both                 1       1       1       chunk overlap recovery boundary
```

Hybrid takes the better rank on every query in this set. The lexical-favorable query that dense had at rank 2 is now at rank 1 because BM25 brought it. The semantic queries that BM25 missed are still rank 1 because dense brought them.

This is the property hybrid was supposed to deliver — the *minimum* of the two retrievers' failure modes, not the *maximum*.

## Step 4: MMR diversification

When a query has high lexical overlap with one document, the top-k
can fill up with chunks from that same document, leaving no room for
diverse perspectives. MMR re-ranks the candidate set to balance
relevance against redundancy.

Algorithm: at each step, pick the candidate that maximizes
`λ * relevance - (1 - λ) * max_similarity_to_already_picked`.

`λ = 0.7` is a reasonable default for "gentle" diversification.

In [ ]:
def mmr_rerank(
    query: str,
    candidates: list[tuple[int, float]],
    lambda_: float = 0.7,
    top_k: int = 5,
) -> list[tuple[int, float]]:
    """Diversify a candidate set via Maximal Marginal Relevance.

    Args:
        query: the query string (used to compute query-doc relevance).
        candidates: [(chunk_idx, _score), ...] ranked-list output from any retriever.
                    The score values are not used; we recompute relevance via dense sim.
        lambda_: relevance/diversity tradeoff. 1.0 = pure relevance,
                 0.5 = aggressive diversification, 0.7 = gentle.
        top_k: number of items to select.

    Returns:
        [(chunk_idx, mmr_score), ...] in selection order.
    """
    if not candidates:
        return []

    query_emb = embedder.encode(
        [query], normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False,
    )[0]

    cand_indices = [c[0] for c in candidates]
    cand_embs = dense_embeddings[cand_indices]  # (n_cands, 384)
    rel_scores = cand_embs @ query_emb  # (n_cands,)

    # Pairwise candidate-candidate sims for redundancy term
    cand_sims = cand_embs @ cand_embs.T  # (n_cands, n_cands)

    selected: list[int] = []          # positions within candidates list
    selected_scores: list[float] = []

    while len(selected) < min(top_k, len(candidates)):
        if not selected:
            best_pos = int(np.argmax(rel_scores))
            mmr_score = float(rel_scores[best_pos])
        else:
            # For each remaining candidate, compute MMR
            remaining = [i for i in range(len(candidates)) if i not in selected]
            max_redundancy = cand_sims[remaining][:, selected].max(axis=1)
            mmr_for_remaining = (
                lambda_ * rel_scores[remaining]
                - (1 - lambda_) * max_redundancy
            )
            best_in_remaining = int(np.argmax(mmr_for_remaining))
            best_pos = remaining[best_in_remaining]
            mmr_score = float(mmr_for_remaining[best_in_remaining])

        selected.append(best_pos)
        selected_scores.append(mmr_score)

    return [(cand_indices[pos], score)
            for pos, score in zip(selected, selected_scores, strict=True)]


# Demo: a query where MMR should change things — a broad term
# matching multiple chunks of the same doc
demo_query = "what is the agent loop"
candidates = hybrid_retrieve(demo_query, top_k=10, candidate_k=30)

print(f"Demo query: {demo_query!r}\n")
print("Before MMR (hybrid top 5):")
for idx, _score in candidates[:5]:
    print(f"  [{all_chunks[idx]['chunk_id']:<30}] {all_chunks[idx]['title']}")

print("\nAfter MMR (λ=0.7, top 5):")
diversified = mmr_rerank(demo_query, candidates, lambda_=0.7, top_k=5)
for idx, _score in diversified:
    print(f"  [{all_chunks[idx]['chunk_id']:<30}] {all_chunks[idx]['title']}")


**Sample output (exact selections may vary):**

```
Demo query: 'what is the agent loop'

Before MMR (hybrid top 5):
  [01-agent-loop.md:0           ] The Agent Loop: A Brief Introduction
  [01-agent-loop.md:1           ] The Agent Loop: A Brief Introduction
  [01-agent-loop.md:2           ] The Agent Loop: A Brief Introduction
  [01-agent-loop.md:3           ] The Agent Loop: A Brief Introduction
  [03-react-pattern.md:0        ] The ReAct Pattern

After MMR (λ=0.7, top 5):
  [01-agent-loop.md:0           ] The Agent Loop: A Brief Introduction
  [03-react-pattern.md:0        ] The ReAct Pattern
  [01-agent-loop.md:2           ] The Agent Loop: A Brief Introduction
  [02-tool-design.md:0          ] Tool Design for Language Model Agents
  [04-search-vs-retrieval.md:0  ] Search and Retrieval: A Useful Distinction
```

Before MMR: 4 of the top 5 were from `01-agent-loop.md` — different chunks of the same document. The model would have to read 4 nearly-overlapping snippets just to see the same content from slightly different angles.

After MMR: still one chunk from `01-agent-loop.md` at the top (highest relevance), then chunks from 4 *different* documents covering related concepts. The agent now has variety to work with.

**When MMR is a no-op:** if the candidate set is already diverse (e.g., on the EVAL_QUERIES from earlier), MMR returns approximately the same order. The overhead is small; the benefit only shows up when the candidates have redundancy.

## Step 5: Cross-encoder reranking

The bi-encoder embeds query and chunk independently. The cross-encoder runs them through the model *together*, capturing fine-grained query-document interactions the bi-encoder can't see.

The cost: ~30-50ms per (query, chunk) pair on CPU. Too slow for a primary retriever, perfect as a reranker on a small candidate set.

We load `cross-encoder/ms-marco-MiniLM-L-6-v2` (~80 MB), rerank the top-30 hybrid candidates, and return the top-5.

In [ ]:
from sentence_transformers import CrossEncoder

print("Loading reranker (cross-encoder/ms-marco-MiniLM-L-6-v2)...")
print("First run downloads ~80 MB; subsequent runs use cache.")
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cpu",
    max_length=512,
)
print(f"Reranker loaded. max_length={reranker.max_length}")


def cross_encoder_rerank(
    query: str,
    candidates: list[tuple[int, float]],
    top_k: int = 5,
) -> list[tuple[int, float]]:
    """Rerank candidate chunks using cross-encoder query-document scoring.

    Returns [(chunk_idx, rerank_score), ...] with rerank_score being the
    cross-encoder's raw logit (higher = more relevant).
    """
    if not candidates:
        return []

    pairs = [(query, all_chunks[idx]["text"]) for idx, _score in candidates]
    rerank_scores = reranker.predict(
        pairs,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

    rescored = list(zip([c[0] for c in candidates],
                        rerank_scores.tolist(), strict=True))
    rescored.sort(key=lambda x: x[1], reverse=True)
    return rescored[:top_k]


# Demo on a query where reranking should pull the right chunk to the top
demo_query = "wrong-but-confident retrieval failure"
candidates = hybrid_retrieve(demo_query, top_k=30, candidate_k=30)

print(f"\nDemo query: {demo_query!r}\n")
print("Hybrid top-5 (before reranking):")
for idx, score in candidates[:5]:
    print(f"  rrf={score:.4f}  [{all_chunks[idx]['chunk_id']:<30}] "
          f"{all_chunks[idx]['text'][:50]}...")

print("\nAfter cross-encoder rerank (top-5 of top-30 candidates):")
reranked = cross_encoder_rerank(demo_query, candidates, top_k=5)
for idx, score in reranked:
    print(f"  rerank={score:7.3f}  [{all_chunks[idx]['chunk_id']:<30}] "
          f"{all_chunks[idx]['text'][:50]}...")


**Sample output (exact scores will vary):**

```
Loading reranker (cross-encoder/ms-marco-MiniLM-L-6-v2)...
First run downloads ~80 MB; subsequent runs use cache.
Reranker loaded. max_length=512

Demo query: 'wrong-but-confident retrieval failure'

Hybrid top-5 (before reranking):
  rrf=0.0164  [04-search-vs-retrieval.md:3 ] IDs. Retrieval is the mechanism behind RAG...
  rrf=0.0161  [02-tool-design.md:1          ] returns a structured result — success with...
  rrf=0.0156  [07-chunking-strategies.md:8 ] The corpus contains duplicates or near-duplicates...
  rrf=0.0156  [04-search-vs-retrieval.md:2 ] used. So why use both? Retrieval gives you...
  rrf=0.0152  [01-agent-loop.md:2           ] by your code, not the model. The tool returns...

After cross-encoder rerank (top-5 of top-30 candidates):
  rerank=  4.821  [04-search-vs-retrieval.md:2 ] used. So why use both? Retrieval gives you...
  rerank=  3.012  [04-search-vs-retrieval.md:3 ] IDs. Retrieval is the mechanism behind RAG...
  rerank=  0.412  [04-search-vs-retrieval.md:0 ] # Search and Retrieval: A Useful Distinction...
  rerank= -1.205  [07-chunking-strategies.md:8 ] The corpus contains duplicates...
  rerank= -2.834  [02-tool-design.md:1          ] returns a structured result — success with...
```

The reranker pulled the most-relevant chunk from `04-search-vs-retrieval.md` to rank 1, with a meaningful score gap (4.8 vs 3.0). The second and third are also from the same document — the *correct* document. The reranker can tell the difference between "this chunk discusses the topic" and "this chunk answers the question" in a way the bi-encoder couldn't.

Note the cross-encoder logits span a much wider range than RRF scores (`-2.8` to `+4.8` here). Bigger separation between good and bad candidates is the reranker's actual job.

## Step 6: The full pipeline

Combine everything: dense + BM25 → RRF → MMR → reranker → top-5.
This is the production-grade retrieval function.

The full pipeline preserves Lab 06's `search_corpus` contract — the
agent loop downstream doesn't change. Only the function body grows.

In [ ]:
MIN_SIMILARITY = 0.0  # We're using rerank logits now — different scale
# (The dense-only floor in Lab 06 was 0.30 against normalized cosines.
# Cross-encoder logits can be negative for non-matches; a sensible floor
# might be `0.0` here. Calibrate per corpus.)


def search_corpus_v2(
    query: str,
    top_k: int = 5,
    candidate_k: int = 30,
    mmr_lambda: float = 0.7,
    use_mmr: bool = False,  # MMR off by default; turn on for redundancy-prone queries
) -> dict:
    """Production-grade search: dense + BM25 → RRF → (optional MMR) → rerank → top-k.

    Same contract as Lab 06's search_corpus:
        ok    → {"status": "ok",    "results": [{chunk_id, doc_id, title, snippet, score, retrieval_signals}, ...]}
        empty → {"status": "empty", "query": ..., "detail": ...}
        error → {"status": "error", "kind": ..., "detail": ...}
    """
    if not query or not query.strip():
        return {"status": "error", "kind": "other", "detail": "empty query"}

    # 1. Dense + BM25 → RRF
    dense_results = dense_retrieve(query, top_k=candidate_k)
    bm25_results = bm25_retrieve(query, top_k=candidate_k)
    fused = reciprocal_rank_fusion(
        {"dense": dense_results, "bm25": bm25_results},
        k=60,
    )[:candidate_k]

    # 2. Optional MMR (off by default — see step 4 commentary)
    if use_mmr:
        fused = mmr_rerank(query, fused, lambda_=mmr_lambda, top_k=candidate_k)

    # 3. Cross-encoder reranking
    reranked = cross_encoder_rerank(query, fused, top_k=top_k)

    # 4. Apply minimum similarity floor (on rerank score)
    above_floor = [(idx, score) for idx, score in reranked
                   if score >= MIN_SIMILARITY]
    if not above_floor:
        top_score = reranked[0][1] if reranked else float("-inf")
        return {
            "status": "empty",
            "query": query,
            "detail": (f"no chunks crossed rerank floor of {MIN_SIMILARITY} "
                      f"(top score was {top_score:.3f})"),
        }

    # 5. Build result envelope. Include the per-retriever signals for debuggability.
    dense_scores = dict(dense_results)
    bm25_scores = dict(bm25_results)

    results = []
    for idx, rerank_score in above_floor:
        chunk = all_chunks[idx]
        snippet = chunk["text"][:200].replace("\n", " ")
        if len(chunk["text"]) > 200:
            snippet += "..."
        results.append({
            "chunk_id": chunk["chunk_id"],
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "snippet": snippet,
            "score": float(rerank_score),
            "retrieval_signals": {
                "dense": dense_scores.get(idx, 0.0),
                "bm25": bm25_scores.get(idx, 0.0),
                "rerank": float(rerank_score),
            },
        })

    return {"status": "ok", "results": results}


# Run the full pipeline on all eval queries
print("─── Full pipeline (dense + BM25 → RRF → rerank, top_k=5) ───")
print(f"{'kind':<20} {'baseline':<10} {'full':<8} query")
print("─" * 90)


def _fmt_rank_v2(r):
    return str(r) if r else "miss"


for q in EVAL_QUERIES:
    # Baseline = Lab 06's dense-only top-5
    baseline = dense_retrieve(q["query"], top_k=5)
    baseline_rank = rank_of_expected_doc(baseline, q["expected_doc"])

    # Full pipeline
    full = search_corpus_v2(q["query"], top_k=5)
    full_ranks = [(c["chunk_id"], c["doc_id"]) for c in full.get("results", [])]
    full_rank = None
    for r, (_cid, doc) in enumerate(full_ranks, start=1):
        if doc == q["expected_doc"]:
            full_rank = r
            break

    print(f"{q['kind']:<20} "
          f"{_fmt_rank_v2(baseline_rank):<10} {_fmt_rank_v2(full_rank):<8} "
          f"{q['query']}")


**Sample output (your exact ranks may vary):**

```
─── Full pipeline (dense + BM25 → RRF → rerank, top_k=5) ───
kind                 baseline   full     query
──────────────────────────────────────────────────────────────────────────────────────────
lexical-favorable    1          1        What is the ReAct pattern?
both                 1          1        agent loop four phases
semantic             1          1        wrong-but-confident retrieval failure
semantic             1          1        what truncates silently in embeddings
lexical-favorable    2          1        structured tool errors discriminated union status
both                 1          1        chunk overlap recovery boundary
```

The pipeline matches or beats the baseline on every query. The clearest win is the lexical-favorable query that BM25 already fixed at the hybrid stage — the reranker then preserves that rank-1 placement.

On this small evaluation set the gains are modest because the baseline was already pretty good. **The pipeline shines on harder queries** — longer, more nuanced, with multiple aspects. The lab's small corpus and easy queries undersell the benefit; on production-grade corpora with thousands of chunks, the precision improvement is often dramatic.

## Step 7: Wire into the agent loop

The whole point of this lab: the agent loop is unchanged. We swap
the body of `search_corpus` and the agent's `read_chunk`-and-cite
pattern keeps working. Same tool contract, same loop, better
retrieval underneath.

This step recreates Lab 06's agent loop and runs the three test
queries from Lab 06 — but using `search_corpus_v2` instead of the
dense-only version.

In [ ]:
# Provider-agnostic chat client (same as Lab 06)
import hashlib


def chat_with_tools(messages: list[dict], tools: list[dict],
                    model: str | None = None) -> dict:
    if PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=model or "gpt-4o-mini",
            messages=messages, tools=tools, temperature=0,
        )
        msg = resp.choices[0].message
        return {
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "name": tc.function.name,
                 "arguments": tc.function.arguments}
                for tc in (msg.tool_calls or [])
            ],
        }
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        anthropic_tools = [
            {"name": t["function"]["name"],
             "description": t["function"]["description"],
             "input_schema": t["function"]["parameters"]}
            for t in tools
        ]
        system = next((m["content"] for m in messages if m["role"] == "system"), None)
        non_system = [m for m in messages if m["role"] != "system"]
        resp = client.messages.create(
            model=model or "claude-haiku-4-5-20251001",
            system=system or "", messages=non_system,
            tools=anthropic_tools, max_tokens=2048,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tool_calls = [
            {"id": b.id, "name": b.name, "arguments": json.dumps(b.input)}
            for b in resp.content if getattr(b, "type", None) == "tool_use"
        ]
        return {"role": "assistant", "content": text, "tool_calls": tool_calls}


# read_chunk — unchanged from Lab 06
def read_chunk(chunk_id: str) -> dict:
    if not chunk_id:
        return {"status": "error", "kind": "other", "detail": "empty chunk_id"}
    chunk = chunks_by_id.get(chunk_id)
    if chunk is None:
        return {"status": "error", "kind": "not_found",
                "detail": f"no chunk with id {chunk_id!r}"}
    return {"status": "ok", "chunk_id": chunk["chunk_id"],
            "doc_id": chunk["doc_id"], "title": chunk["title"],
            "text": chunk["text"]}


# Tool schemas — same contract as Lab 06
TOOLS = [
    {"type": "function", "function": {
        "name": "search_corpus",
        "description": ("Search the corpus by semantic + keyword similarity. "
                       "Returns top-k chunks with snippet, title, and score. "
                       "Phrase queries as 3-8 specific words."),
        "parameters": {"type": "object",
            "properties": {
                "query": {"type": "string"},
                "top_k": {"type": "integer", "description": "1-10, default 5"},
            },
            "required": ["query"],
        }}},
    {"type": "function", "function": {
        "name": "read_chunk",
        "description": ("Read the full text of a chunk by chunk_id. "
                       "Use after search_corpus to inspect a candidate's full content."),
        "parameters": {"type": "object",
            "properties": {"chunk_id": {"type": "string"}},
            "required": ["chunk_id"],
        }}},
]


def execute_tool(name: str, args: dict) -> dict:
    if name == "search_corpus":
        return search_corpus_v2(query=args["query"], top_k=args.get("top_k", 5))
    if name == "read_chunk":
        return read_chunk(chunk_id=args["chunk_id"])
    return {"status": "error", "kind": "other", "detail": f"unknown tool: {name}"}


def _action_hash(name: str, args: dict) -> str:
    payload = name + "|" + json.dumps(args, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


SYSTEM_PROMPT = """You are a research assistant grounded in a specific document corpus.
Answer the user's question only from the corpus. Use search_corpus to find candidates,
then read_chunk to inspect their full text before answering. Phrase queries as 3-8
specific words. Refine if results are poor; do not repeat identical queries.
"""


def run_agent(question: str, max_steps: int = 8, verbose: bool = True) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    citations: list[dict] = []
    seen_actions: set[str] = set()

    for step in range(1, max_steps + 1):
        if verbose:
            print(f"\n── Step {step} ──")

        msg = chat_with_tools(messages, tools=TOOLS)
        entry: dict[str, Any] = {"role": "assistant", "content": msg["content"]}
        if msg["tool_calls"]:
            entry["tool_calls"] = [
                {"id": tc["id"], "type": "function",
                 "function": {"name": tc["name"], "arguments": tc["arguments"]}}
                for tc in msg["tool_calls"]
            ]
        messages.append(entry)

        if not msg["tool_calls"]:
            if verbose:
                print(f"  ◆ FINAL: {msg['content'][:140]}...")
            return {"answer": msg["content"], "citations": citations,
                    "steps": step}

        for tc in msg["tool_calls"]:
            args = json.loads(tc["arguments"]) if tc["arguments"] else {}
            ah = _action_hash(tc["name"], args)
            if ah in seen_actions:
                tool_result = {"status": "error", "kind": "repeated_action",
                              "detail": f"already called {tc['name']} with these args"}
                if verbose:
                    print(f"  ✗ {tc['name']}({args}) [REPEATED]")
            else:
                seen_actions.add(ah)
                tool_result = execute_tool(tc["name"], args)
                if verbose:
                    args_repr = str(args)[:60]
                    print(f"  → {tc['name']}({args_repr}) → {tool_result.get('status')}")
                if tc["name"] == "read_chunk" and tool_result.get("status") == "ok":
                    citations.append({
                        "chunk_id": tool_result["chunk_id"],
                        "doc_id": tool_result["doc_id"],
                        "title": tool_result["title"],
                    })
            messages.append({"role": "tool", "tool_call_id": tc["id"],
                            "content": json.dumps(tool_result)[:4000]})

    return {"answer": "Step cap reached.", "citations": citations,
            "steps": max_steps}


print("Agent ready with the upgraded retrieval pipeline.")


**Sample output:**

```
Agent ready with the upgraded retrieval pipeline.
```

The agent's loop is byte-for-byte the same as Lab 06's. Only `execute_tool` for `search_corpus` calls our new `search_corpus_v2` instead of the dense-only version. The agent doesn't know the retrieval pipeline got more sophisticated — it just gets better results.

In [ ]:
# Run one of Lab 06's harder queries through the upgraded pipeline
query = ("What's the difference between using search as a tool versus using "
         "retrieval as a tool, and what failure modes does each have?")
print(f"QUERY: {query}")
print("=" * 70)
result = run_agent(query, verbose=True)
print("=" * 70)
print(f"\n✓ Steps: {result['steps']}, Citations: {len(result['citations'])}")
print(f"\nAnswer:\n{result['answer']}")
print("\nCitations:")
for c in result["citations"]:
    print(f"  - [{c['chunk_id']}] {c['title']}")


**Sample output:**

```
QUERY: What's the difference between using search as a tool versus using retrieval as a tool, and what failure modes does each have?
======================================================================

── Step 1 ──
  → search_corpus({'query': 'search tool retrieval tool difference failure modes'}) → ok

── Step 2 ──
  → read_chunk({'chunk_id': '04-search-vs-retrieval.md:0'}) → ok

── Step 3 ──
  → read_chunk({'chunk_id': '04-search-vs-retrieval.md:2'}) → ok
  ◆ FINAL: Search and retrieval are two patterns that share an interface...
======================================================================

✓ Steps: 3, Citations: 2

Answer:
Search and retrieval are two patterns that share an interface (the agent
calls a function and gets back ranked text snippets) but differ structurally
on three axes. First, *control*: search queries a corpus you don't control
(the open web)...

Citations:
  - [04-search-vs-retrieval.md:0] Search and Retrieval: A Useful Distinction
  - [04-search-vs-retrieval.md:2] Search and Retrieval: A Useful Distinction
```

Three steps, two citations from the relevant document. In Lab 06 this same query took 5 steps and reached three citations across two different documents — more search-and-refine iterations because the bi-encoder's first hit wasn't as decisive.

The pipeline didn't make the agent smarter. It made the *first retrieval* surface higher-precision candidates, so the agent needed fewer recovery steps.

## Step 8 (stretch): Calibration on your own queries

The defaults in this lab (`candidate_k=30`, `top_k=5`, `λ=0.7`,
`k=60`) work well across many corpora. But you don't have to take
that on faith. Here's the pattern for measuring them on your own
corpus.

1. **Build a validation set.** 10-20 queries where you know the
   correct top chunks. Don't peek at retrieval results when picking
   correct chunks — you want ground truth, not "what the retriever
   liked."

2. **Sweep `candidate_k`.** Run the bi-encoder at `candidate_k = 5,
   10, 20, 40, 80`. Measure: at what `candidate_k` is the correct
   chunk present in 90%+ of queries? That's your minimum useful
   `candidate_k`. Above that, you're spending reranker compute for
   nothing.

3. **Sweep MMR `λ`** if you have queries with redundant top-k. For
   each, count how many distinct documents appear in the top-5. If
   it's always 1-2, MMR with `λ=0.5-0.7` will help. If it's already
   3-5, MMR is a no-op.

4. **Measure rerank lift.** With everything else fixed, compare
   "RRF top-5 directly" vs. "RRF top-30 → rerank → top-5." Count
   how often the correct chunk's *rank* improves. The cost is the
   reranker latency; if it doesn't move the needle on your
   queries, save the compute.

5. **Don't tune `RRF k`.** The Cormack paper's `k=60` is genuinely
   robust; tuning it is rarely worth the work. Start there.

Below is a sketch you can adapt to your own validation set.

In [ ]:
# Sketch: a calibration helper. Adapt the validation_set to your own queries
# and expected chunk IDs (not just docs). For brevity we reuse EVAL_QUERIES.

def evaluate_pipeline(
    validation_set: list[dict],
    retriever_fn,
    name: str,
) -> dict:
    """Run a retriever on each validation query; measure rank of expected doc."""
    ranks = []
    misses = 0
    for q in validation_set:
        results = retriever_fn(q["query"])
        rank = None
        for r, item in enumerate(results, start=1):
            if isinstance(item, dict):  # search_corpus_v2 envelope
                if item["doc_id"] == q["expected_doc"]:
                    rank = r
                    break
            else:  # (idx, score) tuple
                idx, _score = item
                if all_chunks[idx]["doc_id"] == q["expected_doc"]:
                    rank = r
                    break
        if rank is None:
            misses += 1
        else:
            ranks.append(rank)

    return {
        "name": name,
        "hits": len(ranks),
        "misses": misses,
        "mean_rank": float(np.mean(ranks)) if ranks else None,
        "median_rank": float(np.median(ranks)) if ranks else None,
        "all_at_top_1": sum(1 for r in ranks if r == 1),
    }


# Compare every variant
variants = [
    ("dense-only (Lab 06)", lambda q: dense_retrieve(q, top_k=5)),
    ("bm25-only", lambda q: bm25_retrieve(q, top_k=5)),
    ("hybrid (RRF)", lambda q: hybrid_retrieve(q, top_k=5)),
    ("hybrid + rerank", lambda q: search_corpus_v2(q, top_k=5)["results"]
                                  if search_corpus_v2(q, top_k=5)["status"] == "ok"
                                  else []),
]

print(f"{'variant':<25} {'hits':<6} {'@top-1':<8} {'mean rank':<10}")
print("─" * 60)
for name, fn in variants:
    metrics = evaluate_pipeline(EVAL_QUERIES, fn, name)
    mean_r = f"{metrics['mean_rank']:.2f}" if metrics["mean_rank"] else "n/a"
    print(f"{name:<25} {metrics['hits']:<6} "
          f"{metrics['all_at_top_1']:<8} {mean_r:<10}")


**Sample output:**

```
variant                   hits   @top-1   mean rank
────────────────────────────────────────────────────────────
dense-only (Lab 06)       6      5        1.17
bm25-only                 4      4        1.00
hybrid (RRF)              6      6        1.00
hybrid + rerank           6      6        1.00
```

On this tiny eval set, hybrid alone already gets 6/6 at rank 1; reranking can't improve a ranking that's already perfect. The story changes on bigger, harder corpora — on production workloads with longer queries and more chunks, reranking typically lifts mean rank by 1-3 positions and improves @top-1 by 5-15 percentage points.

The discipline matters more than the numbers: **always measure on your own corpus before committing to a pipeline shape.** Defaults are starting points, not answers.

## ✓ Lab complete

You've built a production-grade retrieval pipeline from scratch:

- **Dense retrieval** via the bi-encoder from Lab 06.
- **BM25 retrieval** with `rank-bm25` for lexical matches.
- **Reciprocal Rank Fusion** (~10 lines) to combine retrievers without
  caring about score scales.
- **MMR diversification** (~15 lines of numpy) for redundancy-prone queries.
- **Cross-encoder reranking** with `cross-encoder/ms-marco-MiniLM-L-6-v2`
  to sharpen the top of the ranking.

The whole pipeline lives behind the same `search_corpus` contract Lab 06
established. The agent loop didn't change. Retrieval results did.

Three properties to keep in mind:

1. **Reranking can't surface what retrieval missed.** Set `candidate_k`
   wider than your final `top_k` (5-10× ratio is standard).
2. **Hybrid widens recall; reranking sharpens precision.** They compose;
   neither is a substitute for the other.
3. **Defaults are starting points.** The `λ`, `k`, `candidate_k`, and
   floor values that work on a 55-chunk lab corpus may not generalize.
   Calibrate on your own validation set.

### What to do next

- 🧠 **Take the quiz:** [`quizzes/agentic-rag/retrieval-strategies.md`](../../quizzes/agentic-rag/retrieval-strategies.md)

- 🧭 **Optional: extend this lab.** A few directions:
  1. **Per-query MMR.** Detect redundant top-k and turn on MMR
     selectively instead of globally.
  2. **Score-floor calibration.** Run a batch of off-corpus queries
     and on-corpus queries; pick `MIN_SIMILARITY` empirically.
  3. **Use a hosted reranker.** Swap the cross-encoder for Cohere
     Rerank or Voyage AI rerank — same loop, different latency
     and quality profile.
  4. **Persistent indexes.** `np.save` the dense embeddings and
     `pickle.dump` the BM25 index so subsequent runs skip rebuild.

- 🧭 **Continue Path 02** with the next batch (when it lands):
  contextual retrieval, query expansion, RAG evaluation, framework
  bridge.

- 🧭 **Or move on:** Path 03 (Multi-Agent Systems) or Path 06
  (Evaluation & Observability). Both build on what you've learned.